#### Field Data QA/QC and Biomass Carbon Estimation Pipeline

**Project**: Dryland Carbon MRV Pipeline & Biomass Accounting Engine  
**Methodology**: Verra VM0047 ARR (Afforestation, Reforestation & Revegetation)  
**Purpose**: Demonstrate professional data ingestion, QA/QC with real-world errors, and carbon stock calculation for high-integrity community carbon projects.

#### Setup & Imports

In [1]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

# Ensure correct directory
if "notebooks" in os.getcwd():
    os.chdir("..")

from src.config import *

print("Current Directory:", os.getcwd())
print("Setup completed")

Configuration loaded successfully
Project Root: carbon-mrv-dryland-pipeline
Current Directory: C:\Users\tohiba\Desktop\carbon-mrv-dryland-pipeline
Setup completed


#### Create Realistic Field Dataset with Injected Errors

In [4]:
np.random.seed(42)
n_trees = 28

data = {
    'plot_id': np.random.choice(['P001', 'P002', 'P003', 'P004', 'P005', 'P006'], n_trees, p=[0.25, 0.2, 0.18, 0.15, 0.12, 0.1]),
    'tree_id': [f"T{i:03d}" for i in range(1, n_trees+1)],
    'species': np.random.choice([
        'Acacia tortilis', 'Acacia etbaica', 'Commiphora africana', 
        'Acacia nilotica', 'Acacia senegal', 'Balanites aegyptiaca', 'Ziziphus spina-christi'
    ], n_trees),
    'dbh_cm': np.round(np.random.normal(18, 8, n_trees), 1),
    'height_m': np.round(np.random.normal(6.5, 2.5, n_trees), 1),
    'latitude': np.round(np.random.normal(12.78, 0.08, n_trees), 6),
    'longitude': np.round(np.random.normal(36.28, 0.08, n_trees), 6),
    'collection_date': np.random.choice(['2024-03-10', '2024-03-11', '2024-03-12', '2024-03-15'], n_trees),
    'collector_name': np.random.choice([
        'Abebe Kebede', 'Fatuma Ali', 'Solomon Tesfaye', 
        'Mulugeta Bekele', 'Abebe Kebede', 'Fatuma Ali'
    ], n_trees),
    'notes': ['']*n_trees
}

df_raw = pd.DataFrame(data)

# === Inject realistic field errors (this is very important for portfolio) ===
df_raw.loc[3, 'dbh_cm'] = 0.8          # Impossible small DBH
df_raw.loc[7, 'height_m'] = 28.5       # Unrealistic height
df_raw.loc[12, 'dbh_cm'] = np.nan      # Missing value
df_raw.loc[15, 'latitude'] = 13.45     # Wrong coordinate (out of area)
df_raw.loc[18, 'species'] = 'Acacia Tortilis'   # Capitalization / spelling error
df_raw.loc[22, 'plot_id'] = 'P001'     # Duplicate plot recording error
df_raw.loc[25, 'notes'] = 'Tree damaged by livestock'

# Save as raw data (exactly how it comes from field teams)
df_raw.to_csv(DATA_RAW / "field_inventory_raw.csv", index=False)

print(f"Realistic raw dataset created: {len(df_raw)} trees from {df_raw['plot_id'].nunique()} plots")
print("Real-world errors injected (missing values, outliers, typos, coordinate errors, etc.)")

df_raw.head(5)

Realistic raw dataset created: 28 trees from 6 plots
Real-world errors injected (missing values, outliers, typos, coordinate errors, etc.)


,plot_id,tree_id,species,dbh_cm,height_m,latitude,longitude,collection_date,collector_name,notes
0,P002,T001,Acacia tortilis,16.2,8.6,12.755257,36.307186,2024-03-15,Abebe Kebede,
1,P006,T002,Commiphora africana,21.1,7.2,12.833850,36.279410,2024-03-15,Mulugeta Bekele,
2,P004,T003,Acacia senegal,7.9,3.9,12.759470,36.341384,2024-03-11,Solomon Tesfaye,
3,P003,T004,Commiphora africana,0.8,6.3,12.750574,36.188017,2024-03-15,Abebe Kebede,
4,P001,T005,Ziziphus spina-christi,40.2,8.9,12.881899,36.217973,2024-03-10,Mulugeta Bekele,


#### Professional QA/QC Function

In [7]:
def professional_qa_qc(df):
    """Industry-standard QA/QC pipeline for forest inventory data (VM0047 MRV)"""
    df = df.copy()
    
    # === 1. Basic Flags ===
    df['flag_missing'] = df[['dbh_cm', 'height_m']].isnull().any(axis=1)
    
    # === 2. Value Range Flags ===
    df['flag_dbh'] = (df['dbh_cm'] < 2) | (df['dbh_cm'] > 200)
    df['flag_height'] = (df['height_m'] < 1) | (df['height_m'] > 25)
    
    # === 3. Logical Consistency Flags ===
    df['flag_logic'] = df['height_m'] < (df['dbh_cm'] * 0.12)   # Tree height should be reasonable relative to DBH
    
    # === 4. Coordinate Validation ===
    df['flag_coord'] = (df['latitude'] < 9) | (df['latitude'] > 15) | \
                       (df['longitude'] < 34) | (df['longitude'] > 38)
    
    # === 5. Extreme Values Flag ===
    df['flag_extreme'] = (df['dbh_cm'] > 80) | (df['height_m'] > 18)
    
    # === Overall Flag ===
    flag_cols = ['flag_missing', 'flag_dbh', 'flag_height', 'flag_logic', 'flag_coord', 'flag_extreme']
    df['flag_total'] = df[flag_cols].any(axis=1)
    
    # === QA/QC Summary Report ===
    print("PROFESSIONAL QA/QC REPORT (High-Integrity MRV Standard)")
    print("="*50)
    print(f"Total Trees Recorded          : {len(df)}")
    print(f"Total Flagged Records         : {df['flag_total'].sum()} ({df['flag_total'].mean()*100:.1f}%)")
    print(f"Missing Values                : {df['flag_missing'].sum()}")
    print(f"Invalid DBH                   : {df['flag_dbh'].sum()}")
    print(f"Invalid Height                : {df['flag_height'].sum()}")
    print(f"Logical Inconsistencies       : {df['flag_logic'].sum()}")
    print(f"Coordinate Errors             : {df['flag_coord'].sum()}")
    print(f"Extreme Values                : {df['flag_extreme'].sum()}")
    print("="*70)
    
    # Show flagged records
    if df['flag_total'].sum() > 0:
        print(f"\n Flagged Records ({df['flag_total'].sum()}):")
        display_cols = ['plot_id', 'tree_id', 'species', 'dbh_cm', 'height_m', 'latitude', 'longitude'] + flag_cols
        print(df[df['flag_total']][display_cols].to_string(index=False))
    
    return df


# Apply QA/QC
df_qc = professional_qa_qc(df_raw)

PROFESSIONAL QA/QC REPORT (High-Integrity MRV Standard)
Total Trees Recorded          : 28
Total Flagged Records         : 4 (14.3%)
Missing Values                : 1
Invalid DBH                   : 1
Invalid Height                : 1
Logical Inconsistencies       : 1
Coordinate Errors             : 0
Extreme Values                : 1

 Flagged Records (4):
plot_id tree_id                species  dbh_cm  height_m  latitude  longitude  flag_missing  flag_dbh  flag_height  flag_logic  flag_coord  flag_extreme
   P003    T004    Commiphora africana     0.8       6.3 12.750574  36.188017         False      True        False       False       False         False
   P005    T008 Ziziphus spina-christi    25.1      28.5 12.807641  36.390721         False     False         True       False       False          True
   P005    T013   Balanites aegyptiaca     NaN       3.0 12.797195  36.240525          True     False        False       False       False         False
   P003    T018         Acac

#### Biomass & Carbon Calculation

In [8]:
def calculate_biomass_carbon(df):
    """Calculate Above Ground Biomass (AGB), Carbon Stock and CO₂e 
    using dryland-appropriate allometric equations"""
    
    df = df.copy()
    
    # Load parameters from .env (best practice)
    a = float(os.getenv("ALLOMETRIC_A", 0.0673))      # Coefficient for dryland Acacia/Commiphora
    b = float(os.getenv("ALLOMETRIC_B", 2.63))        # Exponent
    carbon_fraction = float(os.getenv("CARBON_FRACTION", 0.47))
    co2_conversion = float(os.getenv("CO2_CONVERSION", 3.67))
    
    # === Core Calculations ===
    df['agb_kg'] = a * (df['dbh_cm'] ** b)                    # Above Ground Biomass in kg/tree
    df['agb_ton'] = df['agb_kg'] / 1000                       # Convert to tons
    
    df['carbon_ton'] = df['agb_ton'] * carbon_fraction        # Carbon stock (ton/tree)
    df['co2e_ton'] = df['carbon_ton'] * co2_conversion        # CO₂ equivalent (ton/tree)
    
    # Add calculation metadata
    df['calculation_date'] = datetime.now().strftime("%Y-%m-%d %H:%M")
    df['allometric_equation'] = f"AGB = {a} × DBH^{b} (Dryland Ethiopia)"
    
    print("Biomass and Carbon Calculation Completed")
    print(f"   • Allometric Equation : AGB = {a} × DBH^{b}")
    print(f"   • Carbon Fraction     : {carbon_fraction}")
    print(f"   • Total CO₂e          : {df['co2e_ton'].sum():.3f} ton")
    
    return df


# Apply calculation
df_final = calculate_biomass_carbon(df_qc)

# Display key columns
display_cols = ['plot_id', 'tree_id', 'species', 'dbh_cm', 'height_m', 
                'agb_ton', 'carbon_ton', 'co2e_ton', 'flag_total']
df_final[display_cols].round(4).head(12)

Biomass and Carbon Calculation Completed
   • Allometric Equation : AGB = 0.0673 × DBH^2.63
   • Carbon Fraction     : 0.47
   • Total CO₂e          : 9.169 ton


,plot_id,tree_id,species,dbh_cm,height_m,agb_ton,carbon_ton,co2e_ton,flag_total
0,P002,T001,Acacia tortilis,16.2,8.6,0.1021,0.0480,0.1761,False
1,P006,T002,Commiphora africana,21.1,7.2,0.2046,0.0962,0.3529,False
2,P004,T003,Acacia senegal,7.9,3.9,0.0154,0.0073,0.0266,False
3,P003,T004,Commiphora africana,0.8,6.3,0.0000,0.0000,0.0001,True
4,P001,T005,Ziziphus spina-christi,40.2,8.9,1.1146,0.5239,1.9226,False
5,P001,T006,Acacia senegal,27.5,8.5,0.4106,0.1930,0.7083,False
6,P001,T007,Acacia tortilis,19.7,10.2,0.1708,0.0803,0.2946,False
7,P005,T008,Ziziphus spina-christi,25.1,28.5,0.3230,0.1518,0.5571,True
8,P003,T009,Acacia etbaica,9.9,14.9,0.0280,0.0131,0.0482,False
9,P004,T010,Acacia nilotica,5.3,4.2,0.0054,0.0025,0.0093,False


#### Data Cleaning, Final Processing & MRV-Ready Export

In [9]:
def clean_and_export_data(df):
    """Final cleaning and export for MRV purposes"""
    df = df.copy()
    
    # 1. Create Clean Dataset
    df_clean = df[~df['flag_total']].copy()
    
    # 2. Plot Level Summary
    plot_summary = df_clean.groupby('plot_id').agg({
        'tree_id': 'count',
        'agb_ton': ['sum', 'mean'],
        'carbon_ton': ['sum', 'mean'],
        'co2e_ton': ['sum', 'mean'],
        'dbh_cm': 'mean',
        'height_m': 'mean'
    }).round(4)
    
    plot_summary.columns = ['tree_count', 'agb_ton_total', 'agb_ton_avg', 
                           'carbon_ton_total', 'carbon_ton_avg', 
                           'co2e_ton_total', 'co2e_ton_avg', 
                           'mean_dbh_cm', 'mean_height_m']
    
    # 3. Project Level Summary 
    project_summary = {
        'Total_Trees': len(df_clean),
        'Total_Plots': df_clean['plot_id'].nunique(),
        'Total_AGB_ton': round(df_clean['agb_ton'].sum(), 4),
        'Total_Carbon_ton': round(df_clean['carbon_ton'].sum(), 4),
        'Total_CO2e_ton': round(df_clean['co2e_ton'].sum(), 4),
        'Average_Carbon_per_Plot_ton': round(df_clean.groupby('plot_id')['carbon_ton'].sum().mean(), 4),
        'Flagged_Records_Removed': len(df) - len(df_clean),
        'Processing_Date': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        'Methodology': 'Verra VM0047 ARR'
    }
    
    # 4. Export Files 
    # Tree level (full data with flags)
    df.to_csv(DATA_PROCESSED / "tree_level_full_with_flags.csv", index=False)
    
    # Clean tree level
    df_clean.to_csv(DATA_PROCESSED / "tree_level_clean.csv", index=False)
    
    # Plot level summary
    plot_summary.to_csv(DATA_PROCESSED / "plot_level_summary.csv")
    
    # Project summary
    pd.DataFrame([project_summary]).to_csv(DATA_PROCESSED / "project_mrv_summary.csv", index=False)
    
    # Excel Report (Very useful for carbon projects)
    with pd.ExcelWriter(DATA_PROCESSED / "MRV_Full_Report.xlsx") as writer:
        df.to_excel(writer, sheet_name='Tree_Level_With_Flags', index=False)
        df_clean.to_excel(writer, sheet_name='Tree_Level_Clean', index=False)
        plot_summary.to_excel(writer, sheet_name='Plot_Level_Summary')
        pd.DataFrame([project_summary]).to_excel(writer, sheet_name='Project_Summary', index=False)
    
    print("Data Cleaning & Export Completed Successfully!")
    print(f"   • Original Records     : {len(df)}")
    print(f"   • Clean Records        : {len(df_clean)}")
    print(f"   • Files saved to       : {DATA_PROCESSED}")
    print(f"   • Excel Report created : MRV_Full_Report.xlsx")
    
    return df_clean, plot_summary, project_summary

# Run the function
df_clean, plot_summary, project_summary = clean_and_export_data(df_final)

# Show final summary
print("\n Final Project Summary")
for key, value in project_summary.items():
    print(f"{key:25}: {value}")

Data Cleaning & Export Completed Successfully!
   • Original Records     : 28
   • Clean Records        : 24
   • Files saved to       : C:\Users\tohiba\Desktop\carbon-mrv-dryland-pipeline\data\processed
   • Excel Report created : MRV_Full_Report.xlsx

 Final Project Summary
Total_Trees              : 24
Total_Plots              : 6
Total_AGB_ton            : 4.6698
Total_Carbon_ton         : 2.1948
Total_CO2e_ton           : 8.055
Average_Carbon_per_Plot_ton: 0.3658
Flagged_Records_Removed  : 4
Processing_Date          : 2026-05-12 17:47:28
Methodology              : Verra VM0047 ARR
